## 1 Kết nối



In [8]:
import socket
import threading
import queue

HOST = '10.50.85.151'
PORT = 5000

client = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
client.settimeout(3)
client.connect((HOST, PORT))
client.settimeout(None)

inbox = queue.Queue()
history = []

print('Client:', client.getsockname(), '->', client.getpeername())

Client: ('10.50.84.24', 58887) -> ('10.50.85.151', 5000)


## 2 Chuẩn bị gửi và nhận



In [9]:
def receive_forever():
    try:
        with client.makefile('rb') as reader:
            for raw in reader:
                if len(raw) > 8193 or not raw.endswith(b'\n'):
                    break
                inbox.put(raw[:-1].decode('utf-8'))
    except (OSError, UnicodeError):
        pass
    finally:
        inbox.put('[Kết nối đã đóng]')

receiver = threading.Thread(target=receive_forever, daemon=True)
receiver.start()

def send(text):
    if not text or any(c in text for c in '\n\r'):
        raise ValueError('Tin nhắn phải có nội dung và nằm trên một dòng')
    
    payload = f"{text}\n".encode('utf-8')
    if len(payload) > 4096:
        raise ValueError('Tin nhắn quá dài')
        
    client.sendall(payload)

history = []

def read_messages():
    while not inbox.empty():
        history.append(inbox.get_nowait())
    
    if not history:
        print('Chưa có tin nhắn')
    else:
        for message in history:
            print(message)


## 3 Gửi tin



In [20]:
send('i am god!')


## 4 Xem tin nhận được

In toàn bộ lịch sử đã nhận trong phiên này. Chạy lại ô này khi muốn xem tin.


In [ ]:
read_messages()


## 5 Đóng client

Chỉ chạy khi kết thúc buổi thực hành.


In [ ]:
try:
    client.shutdown(socket.SHUT_RDWR)
except OSError:
    pass
client.close()
receiver.join(2)
print('Đã đóng client')
